# Digits Recognition — A Memory-Based Lookup Project

A hands-on study notebook. We build a fast retrieval system for the classic `sklearn` digits dataset and learn the ideas along the way.

**What you will learn**

1. The anatomy of the digits dataset.
2. How to explore it with `pandas` DataFrames.
3. How to store every sample for O(1) exact lookup.
4. Why exact match fails on real-world inputs.
5. Nearest neighbor — brute force, KDTree, BallTree.
6. How to combine an exact-match cache with a tree-based fallback.
7. How to measure both **speed** and **accuracy**.

Read the markdown, run the cell, look at the output, move on. Re-run cells with different inputs to build intuition.

## Setup

If something is missing, install with: `pip install numpy pandas matplotlib scikit-learn`

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neighbors import KDTree, BallTree
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=2, suppress=True, linewidth=120)
pd.set_option('display.precision', 3)

## 1. Load the dataset

`load_digits()` returns a Bunch with:

- `data` — feature matrix, shape `(1797, 64)`; each row is a flattened 8x8 image.
- `images` — same data unflattened, shape `(1797, 8, 8)`.
- `target` — labels 0–9.

Pixel values are integers 0–16 (grayscale intensity).

In [ ]:
digits = load_digits()
print(f'data shape:    {digits.data.shape}')
print(f'images shape:  {digits.images.shape}')
print(f'target shape:  {digits.target.shape}')
print(f'target names:  {digits.target_names}')
print(f'pixel range:   [{digits.data.min():.0f}, {digits.data.max():.0f}]')

### What does a single sample look like?

A digit is just 64 numbers. The `images` array gives us the 8x8 grid form for plotting.

In [ ]:
idx = 0
print(f'Label: {digits.target[idx]}')
print(f'Feature vector (64 values):')
print(digits.data[idx])
print('\nAs an 8x8 image:')
print(digits.images[idx].astype(int))

plt.figure(figsize=(2, 2))
plt.imshow(digits.images[idx], cmap='gray_r')
plt.title(f'label = {digits.target[idx]}')
plt.axis('off')
plt.show()

## 2. Explore with pandas

We load everything into a `DataFrame` to get tabular ergonomics — describe, group-by, value-counts. This is just for exploration; the lookup itself doesn't need pandas.

In [ ]:
df = pd.DataFrame(digits.data, columns=[f'px{i:02d}' for i in range(64)])
df['label'] = digits.target
print(f'Rows: {len(df)}, Columns: {df.shape[1]}')
df.head()

### Are the classes balanced?

In [ ]:
counts = df['label'].value_counts().sort_index()
print(counts)

ax = counts.plot.bar(figsize=(6, 3), color='steelblue', edgecolor='black')
ax.set_title('Samples per digit class')
ax.set_xlabel('digit')
ax.set_ylabel('count')
plt.tight_layout()
plt.show()

~180 samples per class — balanced enough to ignore rebalancing tricks.

### Per-pixel statistics

Which pixels actually carry information? Pixels with `std == 0` are dead weight.

In [ ]:
pixel_stats = df.drop(columns='label').describe().T[['mean', 'std', 'min', 'max']]
print(f'Pixels with zero variance (always blank): {(pixel_stats["std"] == 0).sum()}')
pixel_stats.head(8)

## 3. Visualize the dataset

Ten examples per class. The variation across rows is the noise our lookup will have to tolerate.

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(8, 8))
for cls in range(10):
    samples = digits.images[digits.target == cls][:10]
    for j in range(10):
        axes[cls, j].imshow(samples[j], cmap='gray_r')
        axes[cls, j].axis('off')
    axes[cls, 0].set_title(f'{cls}', loc='left', x=-0.5, y=0.2, fontsize=11)
plt.suptitle('Digits dataset — 10 samples per class', y=0.92)
plt.show()

## 4. Build an exact-match memory

**Goal:** given a feature vector we have seen before, return its label instantly.

We use a Python `dict`. The key must be hashable; a `numpy.ndarray` is not, so we convert it to `bytes` (a stable byte representation of the array's contents).

Lookup is **O(1) average** — independent of dataset size.

In [ ]:
exact_memory = {
    row.tobytes(): int(label)
    for row, label in zip(digits.data, digits.target)
}
print(f'Stored {len(exact_memory)} samples in the hash map')

def exact_lookup(arr):
    key = np.asarray(arr, dtype=np.float64).tobytes()
    return exact_memory.get(key)

print('Lookup sample 0:  ', exact_lookup(digits.data[0]))
print('Lookup sample 100:', exact_lookup(digits.data[100]))

In [ ]:
%timeit exact_lookup(digits.data[42])

### Why exact match isn't enough

Real queries are rarely byte-identical to anything stored: cameras have noise, users draw slightly differently, pixel values shift. Watch what happens when we nudge one pixel by a tiny amount:

In [ ]:
sample = digits.data[42].copy()
print(f'Original lookup:    {exact_lookup(sample)}')

sample[0] += 0.001
print(f'After +0.001 nudge: {exact_lookup(sample)}')

`None`. Even a 0.001 change kills the exact match. We need a way to find the **closest** stored sample.

## 5. Nearest neighbor — brute force

For each query, compute distance to every stored sample, return the closest. Simple, correct, doesn't scale.

Distance metric: **Euclidean (L2)**: `dist(a, b) = sqrt(sum((a_i - b_i)^2))`.

In [ ]:
def brute_nn(query, X=digits.data, y=digits.target):
    diffs = X - query
    sq_dists = (diffs * diffs).sum(axis=1)
    best = int(np.argmin(sq_dists))
    return int(y[best]), float(np.sqrt(sq_dists[best])), best

sample = digits.data[42].copy()
sample[0] += 0.001
label, dist, idx = brute_nn(sample)
print(f'brute NN -> label={label}, dist={dist:.4f}, matched index={idx}')

In [ ]:
%timeit brute_nn(digits.data[42])

Still fast — vectorized numpy on 1797 rows is microseconds. Brute force only breaks down at much larger N.

## 6. Speed up with `KDTree`

A KDTree partitions space recursively along axes. Queries are **O(log n)** average for low-to-moderate dimensions. sklearn ships one.

In [ ]:
kdtree = KDTree(digits.data)

def kdtree_nn(query):
    q = np.asarray(query, dtype=np.float64).reshape(1, -1)
    dist, idx = kdtree.query(q, k=1)
    return int(digits.target[idx[0, 0]]), float(dist[0, 0]), int(idx[0, 0])

print(kdtree_nn(sample))
%timeit kdtree_nn(digits.data[42])

### `BallTree` — better for higher dimensions

KDTree gets less helpful as dimensions grow (the *curse of dimensionality*). A BallTree partitions space into nested hyperspheres instead of axis-aligned boxes — often faster past ~20 dimensions.

In [ ]:
balltree = BallTree(digits.data)

def balltree_nn(query):
    q = np.asarray(query, dtype=np.float64).reshape(1, -1)
    dist, idx = balltree.query(q, k=1)
    return int(digits.target[idx[0, 0]]), float(dist[0, 0]), int(idx[0, 0])

%timeit balltree_nn(digits.data[42])

### Head-to-head benchmark

Pick the winner by averaging over 200 random queries.

In [ ]:
def benchmark(fn, label, n=200):
    rng = np.random.default_rng(0)
    queries = digits.data[rng.choice(len(digits.data), n)]
    t0 = time.perf_counter()
    for q in queries:
        fn(q)
    return {'method': label, 'us_per_query': round((time.perf_counter() - t0) * 1e6 / n, 1)}

results = pd.DataFrame([
    benchmark(brute_nn, 'brute force'),
    benchmark(kdtree_nn, 'KDTree'),
    benchmark(balltree_nn, 'BallTree'),
])
results.sort_values('us_per_query')

On this dataset (small N, 64 dims) brute force is usually competitive. The trees win at larger N. **Choose the structure based on the size of your data, not your gut.**

## 7. Combine: hash for exact, tree for nearest

In a real lookup system you do **both**:

1. Try the hash map first — O(1), nearly free.
2. Fall back to the tree only when needed.

Repeated or memorized queries cost nothing; only novel queries pay the tree-traversal cost.

In [ ]:
class DigitMemory:
    def __init__(self, X, y):
        self.X = np.asarray(X, dtype=np.float64)
        self.y = np.asarray(y)
        self.exact = {self.X[i].tobytes(): int(self.y[i]) for i in range(len(self.y))}
        self.tree = BallTree(self.X)

    def query(self, arr):
        q = np.asarray(arr, dtype=np.float64).reshape(-1)
        hit = self.exact.get(q.tobytes())
        if hit is not None:
            return {'label': hit, 'distance': 0.0, 'exact': True}
        dist, idx = self.tree.query(q.reshape(1, -1), k=1)
        return {
            'label': int(self.y[idx[0, 0]]),
            'distance': float(dist[0, 0]),
            'exact': False,
            'matched_index': int(idx[0, 0]),
        }

mem = DigitMemory(digits.data, digits.target)
print('Exact hit :', mem.query(digits.data[42]))
sample = digits.data[42].copy(); sample[0] += 0.001
print('NN fallback:', mem.query(sample))

## 8. Quality evaluation

Speed is one axis. **Accuracy** is the other. We split into train/test, build the memory on the train set, and measure how often the nearest-neighbor lookup returns the correct label on unseen examples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=42, stratify=digits.target
)
mem_eval = DigitMemory(X_train, y_train)

preds = np.array([mem_eval.query(q)['label'] for q in X_test])
accuracy = (preds == y_test).mean()
print(f'Held-out accuracy: {accuracy:.3%}  ({(preds == y_test).sum()} / {len(y_test)})')

Not bad for a 'store everything, look up the closest' approach with no training.

### Per-class accuracy

In [ ]:
per_class = pd.DataFrame({'true': y_test, 'pred': preds})
per_class['correct'] = per_class['true'] == per_class['pred']
per_class.groupby('true')['correct'].mean().rename('accuracy').to_frame()

### Confusion matrix

Rows are true labels, columns are predictions. Off-diagonal entries are mistakes.

In [ ]:
cm = pd.crosstab(
    pd.Series(y_test, name='true'),
    pd.Series(preds, name='pred'),
)
cm

## 9. Live demo — noisy queries

Take a random sample, add Gaussian noise, ask the memory to recover the digit. Original, noisy query, and the nearest neighbor it found are shown side by side.

Re-run the cell to try different samples and noise levels. Increase `noise_level` until the memory starts making mistakes — that's the breaking point.

In [ ]:
rng = np.random.default_rng()  # different every cell run
i = int(rng.integers(0, len(digits.data)))
noise_level = 3.0

query = digits.data[i] + rng.normal(0, noise_level, 64)
result = mem.query(query)
nn_image = digits.images[result['matched_index']]

fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
axes[0].imshow(digits.images[i], cmap='gray_r')
axes[0].set_title(f'original\nlabel={digits.target[i]}')
axes[1].imshow(query.reshape(8, 8), cmap='gray_r')
axes[1].set_title(f'noisy query\nsigma={noise_level}')
axes[2].imshow(nn_image, cmap='gray_r')
axes[2].set_title(f"recovered\nlabel={result['label']}")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(result)
print('Correct?', result['label'] == int(digits.target[i]))

### Stress test: accuracy vs. noise level

How robust is the memory? We sweep through noise levels and measure accuracy on the held-out test set.

In [ ]:
noise_levels = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]
rng = np.random.default_rng(0)
rows = []
for sigma in noise_levels:
    noisy = X_test + rng.normal(0, sigma, X_test.shape)
    preds = np.array([mem_eval.query(q)['label'] for q in noisy])
    rows.append({'noise_sigma': sigma, 'accuracy': (preds == y_test).mean()})

stress = pd.DataFrame(rows)
print(stress)

ax = stress.plot(x='noise_sigma', y='accuracy', marker='o', figsize=(6, 3), legend=False)
ax.set_title('Lookup accuracy vs. injected noise')
ax.set_xlabel('Gaussian noise sigma (pixel units)')
ax.set_ylabel('accuracy')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Recap

What you built:

1. A **hash-map memory** keyed on the byte representation of the feature vector — O(1) exact lookup.
2. A **nearest-neighbor fallback** with three implementations (brute, KDTree, BallTree) and a small benchmark to pick between them.
3. A **combined memory class** that does both, returning whichever answer comes first.
4. A **held-out evaluation** showing the 1-NN baseline is already ~98% accurate.
5. A **stress test** of accuracy vs. injected noise.

Things to try next:

- Replace Euclidean with cosine distance — does accuracy change?
- Reduce dimensions with PCA before indexing — how much faster does NN get, and at what cost?
- Try k-NN (vote among top-k) instead of plain 1-NN — does it help on the noisy stress test?
- Add a 'rejection threshold' — if the nearest distance is above some cutoff, return 'unknown' instead of guessing.
- Compare to a trained model: `sklearn.linear_model.LogisticRegression` on the same train/test split. Which is more accurate? Which is faster at inference?